In [6]:
# Cell 1: Setup and imports
import json
import time
import os
from typing import Dict, List, Tuple
from collections import defaultdict

from src.llm.chatbot import AI_Waiter
from src.tool.rag.retriever import hybrid_search, build_vector_db
from src.tool.orders.database import OrderDB

# Initialize
bot = AI_Waiter()
bot.initialize_system()

# Load menu for ground truth
with open('/home/lequocthinh/Desktop/BackupCODE/pythonCode/AI_Waiver/data/menu.json', 'r', encoding='utf-8') as f:
    menu_data = json.load(f)

menu_items = {item['name'].lower(): item for item in menu_data}

# Load evaluation datasets
def load_evaluation_datasets():
    """Load all three evaluation datasets"""
    base_path = '/home/lequocthinh/Desktop/BackupCODE/pythonCode/AI_Waiver/data/evaluation'
    
    with open(f'{base_path}/test_rag_dataset.json', 'r', encoding='utf-8') as f:
        rag_dataset = json.load(f)
    
    with open(f'{base_path}/test_router_dataset.json', 'r', encoding='utf-8') as f:
        router_dataset = json.load(f)
    
    with open(f'{base_path}/e2e_test_dataset.json', 'r', encoding='utf-8') as f:
        e2e_dataset = json.load(f)
    
    return rag_dataset, router_dataset, e2e_dataset

# Load datasets
rag_dataset, router_dataset, e2e_dataset = load_evaluation_datasets()
print(f"✅ Loaded RAG dataset: {len(rag_dataset['rag_test_cases'])} test cases")
print(f"✅ Loaded Router dataset: {len(router_dataset['router_test_cases'])} test cases")
print(f"✅ Loaded E2E dataset: {len(e2e_dataset['e2e_scenarios'])} scenarios")

✅ Threshold updated to 0.3
[ INFO ] Building Complete Database (Vector + BM25)
✅ Loaded 17 menu items
✅ Loaded 5 restaurant info sections
🔄 Creating vector store...
✅ Vector database saved to /home/lequocthinh/Desktop/BackupCODE/pythonCode/AI_Waiver/data/indexes/faiss_index
🔄 Building BM25 index...
✅ BM25 index saved with 22 documents

                📊 Database Summary:
                • Total documents: 22
                • Menu items: 17
                • Restaurant info: 5
                • Vector store: ✅ Ready
                • BM25 index: ✅ Ready
                • Score threshold: 0.3
                • Normalization: Sigmoid
                        
✅ Threshold updated to 0.3
[System] RAG database is ready
✅ Loaded RAG dataset: 5 test cases
✅ Loaded Router dataset: 9 test cases
✅ Loaded E2E dataset: 5 scenarios


In [7]:
# Cell 2: RAG Retrieval Evaluation
def evaluate_rag_retrieval():
    """Evaluate RAG retrieval quality with quantitative metrics using dataset"""
    
    test_cases = rag_dataset['rag_test_cases']
    
    metrics = {
        "precision_at_k": {1: [], 3: [], 5: []},
        "recall_at_k": {1: [], 3: [], 5: []},
        "mrr": [],
        "ndcg_at_k": {3: [], 5: []},
        "hit_rate": {1: 0, 3: 0, 5: 0},
        "total_queries": len(test_cases)
    }
    
    print("=" * 70)
    print("RAG RETRIEVAL EVALUATION")
    print("=" * 70)
    
    for i, test_case in enumerate(test_cases, 1):
        query = test_case['query']
        expected_items = test_case.get('expected_menu_items', [])
        expected_type = test_case.get('expected_doc_types')
        category = test_case.get('category', 'unknown')
        
        print(f"\n[{i}/{len(test_cases)}] Query: '{query}' (Category: {category})")
        
        # Get search results
        results = hybrid_search(query, k=5, threshold=0.3)
        
        if not results:
            print("  ❌ No results")
            for k in [1, 3, 5]:
                metrics["precision_at_k"][k].append(0.0)
                metrics["recall_at_k"][k].append(0.0)
            metrics["mrr"].append(0.0)
            continue
        
        # Extract retrieved items
        retrieved_items = []
        retrieved_types = []
        for result in results:
            content = result.document.page_content.lower()
            doc_type = result.doc_type
            
            # Check if any menu item appears in content
            for item_name, item_data in menu_items.items():
                if item_name in content or item_data['name'].lower() in content:
                    retrieved_items.append(item_name)
            
            retrieved_types.append(doc_type)
        
        # Calculate metrics for menu items
        if expected_items:
            expected_set = set([e.lower() for e in expected_items])
            retrieved_set = set(retrieved_items)
            
            # Precision@K
            for k in [1, 3, 5]:
                top_k = retrieved_set if len(retrieved_set) <= k else set(list(retrieved_set)[:k])
                precision = len(top_k & expected_set) / k if k <= len(results) else 0
                metrics["precision_at_k"][k].append(precision)
            
            # Recall@K
            for k in [1, 3, 5]:
                top_k = retrieved_set if len(retrieved_set) <= k else set(list(retrieved_set)[:k])
                recall = len(top_k & expected_set) / len(expected_set) if expected_set else 0
                metrics["recall_at_k"][k].append(recall)
            
            # MRR
            reciprocal_rank = 0
            for rank, item in enumerate(retrieved_items, 1):
                if item in expected_set:
                    reciprocal_rank = 1.0 / rank
                    break
            metrics["mrr"].append(reciprocal_rank)
            
            # Hit@K
            for k in [1, 3, 5]:
                top_k_items = set(retrieved_items[:k])
                if top_k_items & expected_set:
                    metrics["hit_rate"][k] += 1
            
            # NDCG@K
            for k in [3, 5]:
                dcg = 0
                for rank, item in enumerate(retrieved_items[:k], 1):
                    if item in expected_set:
                        dcg += 1.0 / (rank * 1.0)
                
                idcg = sum(1.0 / (r * 1.0) for r in range(1, min(len(expected_set), k) + 1))
                ndcg = dcg / idcg if idcg > 0 else 0
                metrics["ndcg_at_k"][k].append(ndcg)
            
            print(f"  Expected: {expected_items}")
            print(f"  Retrieved: {list(retrieved_set)[:5]}")
            print(f"  Precision@5: {metrics['precision_at_k'][5][-1]:.3f}, "
                  f"Recall@5: {metrics['recall_at_k'][5][-1]:.3f}, "
                  f"MRR: {metrics['mrr'][-1]:.3f}")
        
        # Check doc type
        if expected_type:
            if isinstance(expected_type, list):
                found = any(t in retrieved_types for t in expected_type)
            else:
                found = expected_type in retrieved_types
            
            if found:
                print(f"  ✅ Found expected type: {expected_type}")
            else:
                print(f"  ❌ Expected type {expected_type} not found")
    
    # Calculate averages
    print("\n" + "=" * 70)
    print("RAG RETRIEVAL METRICS SUMMARY")
    print("=" * 70)
    
    for k in [1, 3, 5]:
        avg_precision = sum(metrics["precision_at_k"][k]) / len(metrics["precision_at_k"][k]) if metrics["precision_at_k"][k] else 0
        avg_recall = sum(metrics["recall_at_k"][k]) / len(metrics["recall_at_k"][k]) if metrics["recall_at_k"][k] else 0
        hit_rate = metrics["hit_rate"][k] / metrics["total_queries"]
        
        print(f"\nMetrics@K={k}:")
        print(f"  Average Precision: {avg_precision:.3f}")
        print(f"  Average Recall: {avg_recall:.3f}")
        print(f"  Hit Rate: {hit_rate:.3f} ({metrics['hit_rate'][k]}/{metrics['total_queries']})")
    
    avg_mrr = sum(metrics["mrr"]) / len(metrics["mrr"]) if metrics["mrr"] else 0
    print(f"\nMean Reciprocal Rank (MRR): {avg_mrr:.3f}")
    
    for k in [3, 5]:
        avg_ndcg = sum(metrics["ndcg_at_k"][k]) / len(metrics["ndcg_at_k"][k]) if metrics["ndcg_at_k"][k] else 0
        print(f"NDCG@{k}: {avg_ndcg:.3f}")
    
    return metrics

rag_metrics = evaluate_rag_retrieval()

RAG RETRIEVAL EVALUATION

[1/5] Query: 'phở bò' (Category: menu_search)
✅ Threshold updated to 0.3
🔍 Hybrid searching: 'phở bò' (threshold: 0.30)
📊 BM25 Mean Score: 3.3951

Search Results (BM25 | RAG | Combined):
--------------------------------------------------------------------------------------------------------------
1. Món ăn: Phở Bò Đặc Biệt
Giá: 75000 VNĐ
Mô tả: Nước dùng xương bò hầm 2...
   BM25:   6.73 → 0.9655 | RAG: 1.0000 | Combined: 0.9793
   Source: menu | Type: menu_item

2. Giới thiệu chung
Nhà hàng Bếp Việt Sài Gòn là điểm đến ẩm thực giao th...
   BM25:   1.93 → 0.1882 | RAG: 0.7420 | Combined: 0.4097
   Source: restaurant_info | Type: general_info

3. Món ăn: Bò Nướng Đá Cuội
Giá: 185000 VNĐ
Mô tả: Thịt bò thăn Úc thượng...
   BM25:   3.60 → 0.5505 | RAG: 0.0880 | Combined: 0.3655
   Source: menu | Type: menu_item

4. Món ăn: Bò Lúc Lắc Khoai Tây Chiên
Giá: 145000 VNĐ
Mô tả: Thịt bò thăn...
   BM25:   3.56 → 0.5423 | RAG: 0.0588 | Combined: 0.3489
   Source: menu |

In [8]:
# Cell 3: Router/Tool Selection Evaluation
def evaluate_router():
    """Evaluate router accuracy using dataset"""
    
    test_cases = router_dataset['router_test_cases']
    
    print("\n" + "=" * 70)
    print("ROUTER EVALUATION")
    print("=" * 70)
    
    correct_action = 0
    correct_params = 0
    total = len(test_cases)
    
    action_confusion = defaultdict(lambda: defaultdict(int))
    
    for i, test_case in enumerate(test_cases, 1):
        user_input = test_case['user_input']
        expected_action = test_case['expected_action']
        expected_params = test_case.get('expected_params', {})
        context = test_case.get('context')
        category = test_case.get('category', 'unknown')
        
        print(f"\n[{i}/{total}] Input: '{user_input}' (Category: {category})")
        
        # Setup pending order if context provided
        if context and context.get('pending_order'):
            bot.pending_order = context['pending_order']
        else:
            bot.pending_order = None
        
        bot.chat_history = []  # Reset history for each test
        
        # Get router decision
        tool_call, _ = bot._route(user_input)
        got_action = tool_call.get("action") if tool_call else None
        
        # Check action
        if expected_action is None:
            if got_action is None:
                correct_action += 1
                print("  ✅ Correct: No tool (chat mode)")
            else:
                print(f"  ❌ Wrong: Used tool '{got_action}' but expected chat")
                action_confusion["chat"][got_action] += 1
        else:
            if got_action == expected_action:
                correct_action += 1
                print(f"  ✅ Correct action: {expected_action}")
                
                # Check params
                if expected_params:
                    got_params = tool_call.get("params", {})
                    if expected_action == "search":
                        # Fuzzy matching for search queries
                        expected_query = expected_params.get("query", "").lower()
                        got_query = got_params.get("query", "").lower()
                        key_terms = set(expected_query.split())
                        got_terms = set(got_query.split())
                        if len(key_terms & got_terms) >= len(key_terms) * 0.5:
                            correct_params += 1
                            print(f"  ✅ Params match (fuzzy): {got_query}")
                        else:
                            print(f"  ⚠️  Params mismatch: expected '{expected_query}', got '{got_query}'")
                    else:
                        # Exact match for other actions
                        if all(got_params.get(k) == v for k, v in expected_params.items()):
                            correct_params += 1
                            print(f"  ✅ Params match: {got_params}")
                        else:
                            print(f"  ⚠️  Params mismatch: expected {expected_params}, got {got_params}")
            else:
                print(f"  ❌ Wrong action: expected '{expected_action}', got '{got_action}'")
                action_confusion[expected_action][got_action] += 1
        
        bot.pending_order = None
    
    action_accuracy = correct_action / total
    param_accuracy = correct_params / correct_action if correct_action > 0 else 0
    
    print("\n" + "=" * 70)
    print("ROUTER METRICS SUMMARY")
    print("=" * 70)
    print(f"Action Accuracy: {correct_action}/{total} ({action_accuracy*100:.1f}%)")
    print(f"Parameter Accuracy: {correct_params}/{correct_action} ({param_accuracy*100:.1f}%)")
    
    if action_confusion:
        print("\nConfusion Matrix (expected -> predicted):")
        for expected, predicted_dict in action_confusion.items():
            for predicted, count in predicted_dict.items():
                print(f"  {expected} -> {predicted}: {count}")
    
    return {
        "action_accuracy": action_accuracy,
        "param_accuracy": param_accuracy,
        "correct_action": correct_action,
        "correct_params": correct_params,
        "total": total
    }

router_metrics = evaluate_router()


ROUTER EVALUATION

[1/9] Input: 'cho tôi hỏi giá phở bò đặc biệt' (Category: search_query)
  ✅ Correct action: search
  ✅ Params match (fuzzy): giá phở bò đặc biệt

[2/9] Input: 'quán mình mấy giờ đóng cửa?' (Category: info_query)
  ✅ Correct action: search
  ✅ Params match (fuzzy): giờ đóng cửa

[3/9] Input: 'cho tôi 2 phần phở gà' (Category: order_request)
  ✅ Correct action: draft_order
  ✅ Params match: {'item': 'phở gà', 'quantity': 2}

[4/9] Input: 'cho em 1 tô phở bò' (Category: order_request)
  ✅ Correct action: draft_order
  ✅ Params match: {'item': 'phở bò', 'quantity': 1}

[5/9] Input: 'đúng rồi' (Category: order_confirmation)
  ✅ Correct action: confirm_order
  ✅ Params match: {'decision': 'yes'}

[6/9] Input: 'ok chốt đi' (Category: order_confirmation)
  ❌ Wrong action: expected 'confirm_order', got 'None'

[7/9] Input: 'không, thôi' (Category: order_cancellation)
  ❌ Wrong action: expected 'confirm_order', got 'search'

[8/9] Input: 'chào em, hôm nay quán đông không?' (C

In [9]:
# Cell 4: End-to-End Conversation Evaluation
def evaluate_e2e():
    """Evaluate complete conversation flows using dataset"""
    
    # Use test database
    test_db = "test_orders_eval.db"
    if os.path.exists(test_db):
        os.remove(test_db)
    
    scenarios = e2e_dataset['e2e_scenarios']
    
    print("\n" + "=" * 70)
    print("END-TO-END EVALUATION")
    print("=" * 70)
    
    passed = 0
    total = len(scenarios)
    latency_times = []
    
    for i, scenario in enumerate(scenarios, 1):
        scenario_id = scenario.get('scenario_id', f'scenario_{i}')
        name = scenario['name']
        conversation = scenario['conversation']
        expected_order = scenario.get('expected_order')
        success_criteria = scenario.get('success_criteria', [])
        
        print(f"\n[{i}/{total}] Scenario: {name} (ID: {scenario_id})")
        print("-" * 70)
        
        bot.clear_history()
        bot.pending_order = None
        
        scenario_passed = True
        turn_times = []
        
        for turn_data in conversation:
            turn_num = turn_data['turn']
            user_input = turn_data['user']
            expected_behavior = turn_data['expected_bot_behavior']
            expected_tool_call = turn_data.get('expected_tool_call')
            
            start_time = time.time()
            response = bot.chat(user_input)
            elapsed = time.time() - start_time
            turn_times.append(elapsed)
            
            print(f"Turn {turn_num} ({elapsed:.2f}s): User: '{user_input}'")
            print(f"  Bot: {response[:80]}...")
            
            # Check behavior
            if expected_behavior == "should_contain_price":
                if "vnđ" not in response.lower() and "giá" not in response.lower():
                    print("  ❌ Should mention price but didn't")
                    scenario_passed = False
            
            elif expected_behavior == "should_confirm_order":
                if "chốt" not in response.lower() and "đúng không" not in response.lower():
                    print("  ❌ Should confirm order but didn't")
                    scenario_passed = False
            
            elif expected_behavior == "should_suggest":
                # Check if bot suggests dishes
                if len(response) < 20:  # Too short
                    print("  ⚠️  Response too short, might not suggest properly")
            
            elif expected_behavior == "should_place_order":
                # Check database
                db = OrderDB(test_db)
                recent = db.get_recent_orders(limit=1)
                if not recent:
                    print("  ❌ Order not saved to database")
                    scenario_passed = False
                else:
                    order_item = recent[0][1].lower()  # items column
                    if expected_order:
                        expected_item = expected_order["item"].lower()
                        if expected_item not in order_item:
                            print(f"  ⚠️  Order saved but item mismatch: {order_item}")
            
            elif expected_behavior == "should_cancel":
                if bot.pending_order is not None:
                    print("  ❌ Order should be cancelled but still pending")
                    scenario_passed = False
            
            elif expected_behavior in ["should_greet", "should_chat", "should_acknowledge"]:
                # Just check that response is reasonable
                if len(response) < 5:
                    print("  ⚠️  Response seems too short")
        
        avg_latency = sum(turn_times) / len(turn_times) if turn_times else 0
        latency_times.append(avg_latency)
        
        if scenario_passed:
            passed += 1
            print(f"\n✅ PASSED (avg latency: {avg_latency:.2f}s)")
            print(f"   Success criteria: {', '.join(success_criteria)}")
        else:
            print(f"\n❌ FAILED (avg latency: {avg_latency:.2f}s)")
    
    success_rate = passed / total
    avg_latency = sum(latency_times) / len(latency_times) if latency_times else 0
    
    print("\n" + "=" * 70)
    print("E2E METRICS SUMMARY")
    print("=" * 70)
    print(f"Success Rate: {passed}/{total} ({success_rate*100:.1f}%)")
    print(f"Average Latency per Turn: {avg_latency:.2f}s")
    
    # Cleanup
    if os.path.exists(test_db):
        os.remove(test_db)
    
    return {
        "success_rate": success_rate,
        "passed": passed,
        "total": total,
        "avg_latency": avg_latency
    }

e2e_metrics = evaluate_e2e()


END-TO-END EVALUATION

[1/5] Scenario: Search for menu item price (ID: search_price)
----------------------------------------------------------------------
Chat history cleared.

[ Khách ] : cho tôi hỏi giá phở bò đặc biệt
[System] Using tool: 'search' with params: {'query': 'giá phở bò đặc biệt'}
✅ Threshold updated to 0.3
🔍 Hybrid searching: 'giá phở bò đặc biệt' (threshold: 0.30)
📊 BM25 Mean Score: 4.1507

Search Results (BM25 | RAG | Combined):
--------------------------------------------------------------------------------------------------------------
1. Món ăn: Phở Bò Đặc Biệt
Giá: 75000 VNĐ
Mô tả: Nước dùng xương bò hầm 2...
   BM25:  12.84 → 0.9998 | RAG: 1.0000 | Combined: 0.9999
   Source: menu | Type: menu_item

2. Món ăn: Bò Nướng Đá Cuội
Giá: 185000 VNĐ
Mô tả: Thịt bò thăn Úc thượng...
   BM25:   4.80 → 0.6560 | RAG: 0.0000 | Combined: 0.3936
   Source: menu | Type: menu_item

3. Món ăn: Bò Lúc Lắc Khoai Tây Chiên
Giá: 145000 VNĐ
Mô tả: Thịt bò thăn...
   BM25:   4.72 → 

In [10]:
# Cell 5: Overall System Score
def calculate_overall_score(rag_metrics, router_metrics, e2e_metrics):
    """Calculate weighted overall system score"""
    
    # Extract key metrics
    rag_score = rag_metrics.get("precision_at_k", {}).get(5, [])
    rag_avg = sum(rag_score) / len(rag_score) if rag_score else 0
    
    router_score = router_metrics.get("action_accuracy", 0)
    
    e2e_score = e2e_metrics.get("success_rate", 0)
    
    # Weighted average
    overall = (
        rag_avg * 0.35 +
        router_score * 0.35 +
        e2e_score * 0.30
    )
    
    print("\n" + "=" * 70)
    print("OVERALL SYSTEM SCORE")
    print("=" * 70)
    print(f"RAG Precision@5:     {rag_avg:.3f} (weight: 35%)")
    print(f"Router Accuracy:     {router_score:.3f} (weight: 35%)")
    print(f"E2E Success Rate:    {e2e_score:.3f} (weight: 30%)")
    print(f"\n🎯 Overall Score: {overall:.3f} ({overall*100:.1f}%)")
    
    # Save results
    results = {
        "rag": {
            "precision_at_5": rag_avg,
            "recall_at_5": sum(rag_metrics.get("recall_at_k", {}).get(5, [])) / len(rag_metrics.get("recall_at_k", {}).get(5, [])) if rag_metrics.get("recall_at_k", {}).get(5) else 0,
            "mrr": sum(rag_metrics.get("mrr", [])) / len(rag_metrics.get("mrr", [])) if rag_metrics.get("mrr") else 0,
        },
        "router": router_metrics,
        "e2e": e2e_metrics,
        "overall_score": overall
    }
    
    with open("evaluation_results.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    
    print("\n✅ Results saved to evaluation_results.json")
    
    return results

final_results = calculate_overall_score(rag_metrics, router_metrics, e2e_metrics)


OVERALL SYSTEM SCORE
RAG Precision@5:     0.050 (weight: 35%)
Router Accuracy:     0.778 (weight: 35%)
E2E Success Rate:    0.600 (weight: 30%)

🎯 Overall Score: 0.470 (47.0%)

✅ Results saved to evaluation_results.json
